# 03 — Workplace Risk Score Prediction Model
---
**What this notebook does:**
Trains a supervised ML model that takes all **structured OHS inspection fields**
(plus the `SEVERITY_LEVEL` output from Model 1) and predicts a continuous
**risk score between 0 and 100** for the workplace.

**Why a separate model from the NLP model?**
Model 1 reads the text narrative. Model 2 reads the structured fields.
Together they capture the full picture: *what happened* (text) AND
*the compliance context* (order type, order status, sector, case type).
The two models are chained — Model 1's output is an input feature for Model 2.

**Architecture:**
```
Structured fields + SEVERITY_LEVEL (from Model 1)
     │
     ▼
 Feature Engineering
 (date parts, sector, binary flags, label encoding)
     │
     ▼
 XGBoost Regressor (300 trees, learning_rate=0.05)
     │
     ▼
 risk_score (0–100) + risk_category + SHAP explainability
```

**Why XGBoost?**
- Handles mixed feature types (categorical + numeric) natively after encoding
- Captures non-linear interactions (e.g. Mining sector + Stop Work Order = very high risk)
- Built-in feature importance and compatible with SHAP for explainability
- Fast training even on CPU with `tree_method="hist"`

**Output:** Registered MLflow model → `ohs_risk_score_model` (Version 1)

**Runtime:** Serverless &nbsp;|&nbsp; **Prerequisite:** Notebook 01 must have run successfully

## Cell 1 — Install Dependencies
`xgboost` is pre-installed on Databricks ML Runtime but may need a version update.
`shap` is used to compute feature importance explanations for each prediction.
Both are installed here to ensure version consistency.

In [0]:
%pip install xgboost shap --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys
print(f"Current Python version: {sys.version}")

Current Python version: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]


## Cell 2 — Imports
Import all libraries. Key imports:
- `xgboost` — the gradient boosting regressor
- `shap` — SHapley Additive exPlanations for model interpretability
- `mlflow.pyfunc` — for creating a custom serveable model class

In [0]:
import pandas as pd
import numpy as np
import pickle                           # for saving model artifacts

# Scikit-learn components
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost — gradient boosted trees (best-in-class for tabular regression)
import xgboost as xgb

# SHAP — model explainability (shows which features most influenced each prediction)
import shap

# MLflow — experiment tracking, model packaging, and registry
import mlflow
import mlflow.pyfunc

## Cell 3 — Load Training Data
Load the same synthetic dataset used to train Model 1.
For Model 2 we use all structured columns plus `SEVERITY_LEVEL`
(which in production comes from Model 1's output).

In [0]:
# Load from Unity Catalog Delta table (workspace = your catalog name)
df = spark.table("workspace.ohs_data.synthetic_inspections").toPandas()

print(f"Loaded {len(df)} records from workspace.ohs_data.synthetic_inspections")
print(f"\nRisk score distribution (our regression target):")
print(df["RISK_SCORE"].describe().round(2))
print(f"\nRisk score by severity tier:")
print(df.groupby("SEVERITY_LEVEL")["RISK_SCORE"].agg(["mean", "min", "max"]).round(2))

Loaded 100 records from workspace.ohs_data.synthetic_inspections

Risk score distribution (our regression target):
count    100.00
mean      52.49
std       28.21
min        0.00
25%       27.78
50%       49.30
75%       75.52
max      100.00
Name: RISK_SCORE, dtype: float64

Risk score by severity tier:
                 mean   min    max
SEVERITY_LEVEL                    
Critical        90.48  62.9  100.0
High            79.43  60.9  100.0
Low             18.95   0.0   33.4
Medium          46.26  22.9   71.6


## Cell 4 — Feature Engineering Constants and Functions

Raw OHS columns need to be transformed before XGBoost can use them.
This cell defines the feature engineering pipeline:

| Feature type | Raw column | Engineered feature |
|---|---|---|
| Categorical encoding | ORDER_TYPE, CASE_TYPE, etc. | `{COL}_ENC` integer |
| Date decomposition | FIELD_VISIT_DATE | VISIT_YEAR, VISIT_MONTH, QUARTER, DOW |
| Sector derivation | PRIMARY_NAICS | SECTOR (Construction/Mining/Healthcare/Industrial) |
| NAICS grouping | PRIMARY_NAICS | NAICS_GROUP (first 3 digits) |
| Binary flags | ORDER_TYPE, ORDER_STATUS, CASE_TYPE | IS_STOP_WORK, IS_NON_COMPLIANT, etc. |
| Severity encoding | SEVERITY_LEVEL (from Model 1) | SEVERITY_ENCODED (1–4 integer) |

The `engineer_features()` function runs in two modes:
- `fit=True` — builds the label encoders from training data (used here)
- `fit=False` — uses saved encoders to transform new data (used inside the pyfunc at serve time)

In [0]:
# ── Columns that will be label-encoded ──────────────────────────────────────
# These are the categorical string columns from the OHS dataset
CATEGORICAL_COLS = [
    "FIELD_VISIT_TYPE", "CASE_TYPE", "CASE_STATUS",
    "CONTRAVENER_ROLE", "ORDER_TYPE", "ORDER_STATUS",
    "CASE_ACT", "ACT_REG_ID", "SEC",
]

# ── Severity mapping — converts string labels to ordinal integers ─────────────
# Critical=4 is the highest risk, Low=1 is the lowest
# This gives the model a continuous severity signal rather than a one-hot encoding
SEVERITY_MAP = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

# ── Sector risk mapping — for label encoding the derived sector column ─────────
SECTOR_MAP = {"Construction": 0, "Mining": 1, "Health Care": 2, "Industrial": 3}

# ── Numeric flags included in the final feature vector ───────────────────────
NUMERIC_FLAGS = [
    "VISIT_YEAR", "VISIT_MONTH", "VISIT_QUARTER", "VISIT_DOW",  # date features
    "IS_STOP_WORK",         # 1 if order was a Stop Work Order (very high risk)
    "IS_INVESTIGATION",     # 1 if this was a reactive investigation (incident already happened)
    "IS_NON_COMPLIANT",     # 1 if employer has NOT complied with the order
    "IS_CLOSED",            # 1 if the case is already closed
    "IS_HIGH_RISK_ORDER",   # 1 if order type is SWO, Plan Order, or Time Unknown
    "HAS_SUBSEC",           # 1 if a specific subsection was cited (more detailed violation)
    "HAS_CLAUSE",           # 1 if a specific clause was cited (most detailed violation)
    "SEVERITY_ENCODED",     # ordinal encoding of severity from Model 1 (1–4)
    "SECTOR_ENC",           # encoded sector (Construction/Mining/Healthcare/Industrial)
]

# Final feature column list = encoded categoricals + numeric flags
FEATURE_COLS = [f"{c}_ENC" for c in CATEGORICAL_COLS + ["NAICS_GROUP"]] + NUMERIC_FLAGS


def _get_sector(naics: str) -> str:
    """Derive the industry sector from the NAICS code prefix."""
    n = str(naics)
    if n.startswith("23"): return "Construction"
    if n.startswith("21"): return "Mining"
    if n.startswith("6"):  return "Health Care"
    return "Industrial"


def engineer_features(
    df: pd.DataFrame,
    encoders: dict | None = None,
    fit: bool = True,
) -> tuple:
    """
    Full feature engineering pipeline.

    Parameters:
      df       : raw DataFrame with original OHS columns
      encoders : dict of fitted LabelEncoders (required when fit=False)
      fit      : if True, fit new encoders (training mode)
                 if False, use provided encoders (inference mode)

    Returns:
      (transformed_df, encoders_dict)
    """
    df = df.copy()

    # ── Date features — decompose visit date into numeric components ──────────
    df["FIELD_VISIT_DATE"] = pd.to_datetime(df["FIELD_VISIT_DATE"], errors="coerce")
    df["VISIT_YEAR"]    = df["FIELD_VISIT_DATE"].dt.year.fillna(2024).astype(int)
    df["VISIT_MONTH"]   = df["FIELD_VISIT_DATE"].dt.month.fillna(1).astype(int)
    df["VISIT_QUARTER"] = df["FIELD_VISIT_DATE"].dt.quarter.fillna(1).astype(int)
    df["VISIT_DOW"]     = df["FIELD_VISIT_DATE"].dt.dayofweek.fillna(0).astype(int)  # 0=Monday

    # ── Sector and NAICS group — derived from the NAICS code ─────────────────
    df["SECTOR"]      = df["PRIMARY_NAICS"].astype(str).apply(_get_sector)
    df["NAICS_GROUP"] = df["PRIMARY_NAICS"].astype(str).str[:3]   # first 3 digits

    # ── Binary flags — high-signal risk indicators ────────────────────────────
    df["IS_STOP_WORK"]      = (df["ORDER_TYPE"]   == "Stop Work Order").astype(int)
    df["IS_INVESTIGATION"]  = (df["CASE_TYPE"]    == "Investigation").astype(int)
    df["IS_NON_COMPLIANT"]  = (df["ORDER_STATUS"] == "Not Complied With").astype(int)
    df["IS_CLOSED"]         = (df["CASE_STATUS"]  == "Closed").astype(int)
    df["IS_HIGH_RISK_ORDER"] = df["ORDER_TYPE"].isin(
        ["Stop Work Order", "Plan Order", "Time Unknown Order"]
    ).astype(int)
    df["HAS_SUBSEC"] = (df["SUBSEC"].fillna("").str.strip() != "").astype(int)
    df["HAS_CLAUSE"] = (df["CLAUSE"].fillna("").str.strip() != "").astype(int)

    # ── Severity and sector encodings ─────────────────────────────────────────
    # In production, SEVERITY_LEVEL comes from Model 1's prediction
    df["SEVERITY_ENCODED"] = (
        df.get("SEVERITY_LEVEL", pd.Series(["Medium"] * len(df)))
          .map(SEVERITY_MAP).fillna(2).astype(int)
    )
    df["SECTOR_ENC"] = df["SECTOR"].map(SECTOR_MAP).fillna(3).astype(int)

    # ── Label encode all categorical columns ──────────────────────────────────
    all_cat = CATEGORICAL_COLS + ["NAICS_GROUP"]
    if fit:
        # Training mode: fit new encoders and return them
        encoders = {}
        for col in all_cat:
            le = LabelEncoder()
            df[f"{col}_ENC"] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
    else:
        # Inference mode: use pre-fitted encoders; unseen values → first class
        for col in all_cat:
            le   = encoders[col]
            vals = df[col].astype(str).apply(
                lambda x: x if x in le.classes_ else le.classes_[0]
            )
            df[f"{col}_ENC"] = le.transform(vals)

    return df, encoders

## Cell 5 — Apply Feature Engineering and Prepare Feature Matrix

Run the full pipeline on the training data, then extract:
- `X` — the feature matrix (all engineered feature columns as a numpy array)
- `y` — the target vector (RISK_SCORE values)

In [0]:
# Apply feature engineering in fit mode (builds and saves the label encoders)
df_feat, encoders = engineer_features(df, fit=True)

# Extract the feature matrix and target vector
X = df_feat[FEATURE_COLS].values    # shape: (500, n_features)
y = df_feat["RISK_SCORE"].values    # shape: (500,)  — continuous 0–100

print(f"Feature matrix shape : {X.shape}")
print(f"Number of features   : {len(FEATURE_COLS)}")
print(f"Feature names        : {FEATURE_COLS}")
print(f"\nTarget range : {y.min():.1f} – {y.max():.1f}  (mean: {y.mean():.1f})")

# Stratified split not needed for regression — plain random 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} records  |  Test: {len(X_test)} records")

Feature matrix shape : (100, 23)
Number of features   : 23
Feature names        : ['FIELD_VISIT_TYPE_ENC', 'CASE_TYPE_ENC', 'CASE_STATUS_ENC', 'CONTRAVENER_ROLE_ENC', 'ORDER_TYPE_ENC', 'ORDER_STATUS_ENC', 'CASE_ACT_ENC', 'ACT_REG_ID_ENC', 'SEC_ENC', 'NAICS_GROUP_ENC', 'VISIT_YEAR', 'VISIT_MONTH', 'VISIT_QUARTER', 'VISIT_DOW', 'IS_STOP_WORK', 'IS_INVESTIGATION', 'IS_NON_COMPLIANT', 'IS_CLOSED', 'IS_HIGH_RISK_ORDER', 'HAS_SUBSEC', 'HAS_CLAUSE', 'SEVERITY_ENCODED', 'SECTOR_ENC']

Target range : 0.0 – 100.0  (mean: 52.5)

Train: 80 records  |  Test: 20 records


## Cell 6 — Define the Custom MLflow PythonModel

Same pattern as Notebook 02 — we wrap the full pipeline (preprocessing + model)
into a single `mlflow.pyfunc.PythonModel` so the serving endpoint accepts
raw JSON input (no preprocessing needed on the Flask app side).

**The `_preprocess` method inside the class replicates the full feature
engineering pipeline** using the saved encoders, ensuring training-time and
inference-time transformations are always identical.

**Serving endpoint I/O format:**
```
Input:  {"dataframe_records": [{"FIELD_VISIT_DATE": "2024-03-15",
                                 "ORDER_TYPE": "Stop Work Order",
                                 "SEVERITY_LEVEL": "Critical", ...}]}
Output: [{"risk_score": 91.4, "risk_category": "Critical Risk", "risk_level_int": 4}]
```

In [0]:
class RiskScoreModel(mlflow.pyfunc.PythonModel):
    """
    Custom MLflow pyfunc model that bundles:
      - Full feature engineering pipeline (with saved LabelEncoders)
      - XGBoost Regressor trained on engineered features

    Accepts raw OHS inspection fields + SEVERITY_LEVEL (from Model 1).
    Returns risk_score (0-100), risk_category, and risk_level_int.
    """

    def load_context(self, context: mlflow.pyfunc.PythonModelContext) -> None:
        """
        Called once at serving startup.
        Loads the XGBoost model, label encoders, and feature column list
        from the pickled artifact file.
        """
        import pickle
        with open(context.artifacts["model_artifacts"], "rb") as fh:
            arts = pickle.load(fh)

        # Unpack all saved artifacts
        self.model       = arts["model"]          # trained XGBoost regressor
        self.encoders    = arts["encoders"]       # dict of fitted LabelEncoders
        self.feature_cols = arts["feature_cols"]  # ordered list of feature column names
        self.cat_cols    = arts["categorical_cols"]  # list of categorical col names

    def _preprocess(self, df: pd.DataFrame) -> np.ndarray:
        """
        Replicates the feature engineering pipeline from training.
        Called inside predict() before passing data to the XGBoost model.
        Uses the saved encoders to transform unseen data consistently.
        """
        import pandas as pd
        import numpy as np

        df = df.copy()

        # Decompose visit date into numeric components
        df["FIELD_VISIT_DATE"] = pd.to_datetime(df["FIELD_VISIT_DATE"], errors="coerce")
        df["VISIT_YEAR"]    = df["FIELD_VISIT_DATE"].dt.year.fillna(2024).astype(int)
        df["VISIT_MONTH"]   = df["FIELD_VISIT_DATE"].dt.month.fillna(1).astype(int)
        df["VISIT_QUARTER"] = df["FIELD_VISIT_DATE"].dt.quarter.fillna(1).astype(int)
        df["VISIT_DOW"]     = df["FIELD_VISIT_DATE"].dt.dayofweek.fillna(0).astype(int)

        # Derive sector and NAICS group from the NAICS code
        def _sect(n):
            n = str(n)
            if n.startswith("23"): return "Construction"
            if n.startswith("21"): return "Mining"
            if n.startswith("6"):  return "Health Care"
            return "Industrial"

        df["SECTOR"]      = df["PRIMARY_NAICS"].astype(str).apply(_sect)
        df["NAICS_GROUP"] = df["PRIMARY_NAICS"].astype(str).str[:3]

        # Compute all binary flag features
        df["IS_STOP_WORK"]       = (df["ORDER_TYPE"]   == "Stop Work Order").astype(int)
        df["IS_INVESTIGATION"]   = (df["CASE_TYPE"]    == "Investigation").astype(int)
        df["IS_NON_COMPLIANT"]   = (df["ORDER_STATUS"] == "Not Complied With").astype(int)
        df["IS_CLOSED"]          = (df["CASE_STATUS"]  == "Closed").astype(int)
        df["IS_HIGH_RISK_ORDER"] = df["ORDER_TYPE"].isin(
            ["Stop Work Order", "Plan Order", "Time Unknown Order"]
        ).astype(int)
        df["HAS_SUBSEC"] = (
            df.get("SUBSEC", pd.Series([""]*len(df))).fillna("").str.strip() != ""
        ).astype(int)
        df["HAS_CLAUSE"] = (
            df.get("CLAUSE", pd.Series([""]*len(df))).fillna("").str.strip() != ""
        ).astype(int)

        # Encode severity as ordinal integer (1=Low, 2=Medium, 3=High, 4=Critical)
        sev_map = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
        df["SEVERITY_ENCODED"] = (
            df.get("SEVERITY_LEVEL", pd.Series(["Medium"]*len(df)))
              .map(sev_map).fillna(2).astype(int)
        )

        # Encode sector
        sec_map = {"Construction": 0, "Mining": 1, "Health Care": 2, "Industrial": 3}
        df["SECTOR_ENC"] = df["SECTOR"].map(sec_map).fillna(3).astype(int)

        # Label encode all categorical columns using the saved encoders
        # Unseen values (at inference time) are mapped to the first known class
        for col in self.cat_cols + ["NAICS_GROUP"]:
            le   = self.encoders[col]
            vals = df[col].astype(str).apply(
                lambda x: x if x in le.classes_ else le.classes_[0]
            )
            df[f"{col}_ENC"] = le.transform(vals)

        # Return only the features in the same order as training
        return df[self.feature_cols].values.astype(float)

    def predict(
        self,
        context: mlflow.pyfunc.PythonModelContext,
        model_input: pd.DataFrame,
    ):
        """
        Called for every serving request.
        Preprocesses input, runs XGBoost prediction, returns structured output.
        """
        import numpy as np

        # Accept both single-record dict and multi-record DataFrame
        if isinstance(model_input, dict):
            model_input = pd.DataFrame([model_input])

        # Run the full preprocessing pipeline
        X = self._preprocess(model_input)

        # Get raw predictions from XGBoost and clamp to [0, 100]
        raw_scores = self.model.predict(X)

        results = []
        for score in raw_scores:
            score = float(np.clip(score, 0.0, 100.0))

            # Map the numeric score to a risk category and integer level
            if   score >= 75: cat, lvl = "Critical Risk", 4
            elif score >= 55: cat, lvl = "High Risk",     3
            elif score >= 35: cat, lvl = "Medium Risk",   2
            else:             cat, lvl = "Low Risk",      1

            results.append({
                "risk_score":     round(score, 1),   # continuous 0–100
                "risk_category":  cat,               # human-readable tier
                "risk_level_int": lvl,               # integer 1–4 for programmatic use
            })

        return results

## Cell 7 — Train XGBoost and Log to MLflow

Trains the XGBoost regressor with the following key hyperparameters:
- `n_estimators=300` — 300 boosting rounds (trees)
- `learning_rate=0.05` — slow learning rate to prevent overfitting
- `max_depth=6` — maximum tree depth (controls model complexity)
- `subsample=0.8` — use 80% of rows per tree (reduces variance)
- `colsample_bytree=0.8` — use 80% of features per tree (reduces variance)
- `tree_method="hist"` — histogram-based algorithm, fast on CPU

After training, SHAP values are computed to explain which features
matter most for the risk score predictions.

In [0]:
# Set the MLflow experiment — creates it if it doesn't exist
mlflow.set_experiment("/Workspace/Users/nishit.rathod@ontario.ca/SDS_Assignment_243232/risk_score_model")

with mlflow.start_run(run_name="risk_xgb_v1") as run:

    # ── Define hyperparameters ────────────────────────────────────────────────
    params = {
        "n_estimators":      300,    # number of boosting trees
        "max_depth":           6,    # max depth per tree (6 is a good default)
        "learning_rate":    0.05,    # low LR → more trees, better generalisation
        "subsample":         0.8,    # row sampling per tree (reduces overfitting)
        "colsample_bytree":  0.8,    # feature sampling per tree (reduces overfitting)
        "min_child_weight":    3,    # minimum samples in a leaf (prevents tiny splits)
        "random_state":       42,
        "tree_method":     "hist",   # histogram method — fast on CPU and GPU
        "device":           "cpu",   # change to "cuda" for GPU serving endpoint
    }

    # ── Train the model ────────────────────────────────────────────────────────
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],   # monitor test loss during training
        verbose=50,                    # print eval metric every 50 rounds
    )

    # ── Evaluate on the hold-out test set ─────────────────────────────────────
    y_pred = xgb_model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2   = r2_score(y_test, y_pred)

    print(f"\nTest Set Metrics:")
    print(f"  MAE  (mean absolute error)  : {mae:.3f}  pts  ← avg error in risk score units")
    print(f"  RMSE (root mean sq error)   : {rmse:.3f} pts")
    print(f"  R²   (variance explained)   : {r2:.3f}   ← 1.0 = perfect, 0.0 = baseline mean")

    # ── Feature importance (XGBoost built-in) ────────────────────────────────
    feat_imp = (
        pd.DataFrame({"feature": FEATURE_COLS, "importance": xgb_model.feature_importances_})
          .sort_values("importance", ascending=False)
    )
    print("\nTop 15 Most Important Features (XGBoost gain importance):")
    print(feat_imp.head(15).to_string(index=False))

    # ── SHAP values — model-agnostic explainability ───────────────────────────
    # TreeExplainer is the most efficient SHAP method for tree-based models
    # Mean absolute SHAP values show the average impact of each feature across all predictions
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test[:50])   # compute on first 50 test records

    shap_df = (
        pd.DataFrame(
            np.abs(shap_values).mean(axis=0),  # average absolute impact per feature
            index=FEATURE_COLS,
            columns=["mean_abs_shap"],
        ).sort_values("mean_abs_shap", ascending=False)
    )
    print("\nTop 10 Features by SHAP importance (mean absolute SHAP value):")
    print(shap_df.head(10).to_string())

    # ── Log to MLflow ──────────────────────────────────────────────────────────
    mlflow.log_params(params)
    mlflow.log_metric("mae",  mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2",   r2)

    # ── Save all model artifacts to a single pickle file ─────────────────────
    # This file is the only artifact needed by the pyfunc model at serve time
    arts_path = "/tmp/risk_model_artifacts.pkl"
    with open(arts_path, "wb") as fh:
        pickle.dump({
            "model":           xgb_model,        # trained XGBoost regressor
            "encoders":        encoders,          # dict of LabelEncoders from fit step
            "feature_cols":    FEATURE_COLS,      # ordered feature list for inference
            "categorical_cols": CATEGORICAL_COLS, # which cols need label encoding
            "numeric_flags":   NUMERIC_FLAGS,     # which cols are numeric/binary
        }, fh)

    # ── Define the conda environment for the serving container ────────────────
    conda_env = {
        "channels": ["defaults", "conda-forge"],
        "dependencies": [
            "python=3.12", "pip",
            {"pip": [
                "xgboost>=1.7.0",
                "scikit-learn>=1.2.0",
                "shap>=0.42.0",
                "pandas>=1.5.0",
                "numpy>=1.23.0",
                "mlflow>=2.0.0",
            ]},
        ],
        "name": "ohs_risk_env",
    }

    # ── Create model signature for Unity Catalog registration ─────────────────
    # Unity Catalog requires both input and output types to be specified
    # Generate sample prediction using raw input (pre-feature-engineering)
    sample_input = df_feat.iloc[:1][[
        "FIELD_VISIT_DATE", "FIELD_VISIT_TYPE", "CASE_TYPE", "CASE_STATUS",
        "CONTRAVENER_ROLE", "ORDER_TYPE", "ORDER_STATUS", "CASE_ACT",
        "ACT_REG_ID", "SEC", "SUBSEC", "CLAUSE", "PRIMARY_NAICS",
        "SEVERITY_LEVEL"  # from Model 1
    ]].copy()
    sample_output = [{
        "risk_score": 75.3,
        "risk_category": "Critical Risk",
        "risk_level_int": 4
    }]
    signature = mlflow.models.infer_signature(sample_input, sample_output)

    # ── Log the pyfunc model with the bundled artifacts ───────────────────────
    mlflow.pyfunc.log_model(
        artifact_path="risk_model",
        python_model=RiskScoreModel(),

        # Only one artifact needed — the pickle contains model + encoders together
        artifacts={"model_artifacts": arts_path},

        conda_env=conda_env,

        signature=signature,              # Required for Unity Catalog registration
        input_example=sample_input,       # Shows expected input format in UI

        # Register in the MLflow Model Registry
        registered_model_name="ohs_risk_score_model",
    )

    print(f"\n✓  MLflow run ID : {run.info.run_id}")
    print(f"✓  Registered as : ohs_risk_score_model (Version 1)")
    print(f"   → Go to Machine Learning → Models to see it")

2026/05/31 19:40:41 INFO mlflow.tracking.fluent: Experiment with name '/Workspace/Users/nishit.rathod@ontario.ca/SDS_Assignment_243232/risk_score_model' does not exist. Creating a new experiment.


[0]	validation_0-rmse:25.00392
[50]	validation_0-rmse:13.72931
[100]	validation_0-rmse:12.55923
[150]	validation_0-rmse:12.40370
[200]	validation_0-rmse:12.36755
[250]	validation_0-rmse:12.36252
[299]	validation_0-rmse:12.35838

Test Set Metrics:
  MAE  (mean absolute error)  : 9.854  pts  ← avg error in risk score units
  RMSE (root mean sq error)   : 12.358 pts
  R²   (variance explained)   : 0.772   ← 1.0 = perfect, 0.0 = baseline mean

Top 15 Most Important Features (XGBoost gain importance):
             feature  importance
    SEVERITY_ENCODED    0.388347
  IS_HIGH_RISK_ORDER    0.191044
        IS_STOP_WORK    0.122522
          HAS_CLAUSE    0.075232
    IS_NON_COMPLIANT    0.038241
       CASE_TYPE_ENC    0.022277
FIELD_VISIT_TYPE_ENC    0.020421
    ORDER_STATUS_ENC    0.015552
      ORDER_TYPE_ENC    0.014009
     CASE_STATUS_ENC    0.011967
CONTRAVENER_ROLE_ENC    0.011364
           IS_CLOSED    0.011207
       VISIT_QUARTER    0.010665
           VISIT_DOW    0.010664
   

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/31 19:40:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-71dc0835-779f.cloud.databricks.com/ml/experiments/3583728162249090/models/m-54915450c095424eb6fd5

Successfully registered model 'workspace.default.ohs_risk_score_model'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.ohs_risk_score_model': https://dbc-71dc0835-779f.cloud.databricks.com/explore/data/models/workspace/default/ohs_risk_score_model/version/1?o=3652072476397248



✓  MLflow run ID : 553e0401818143f487b4ef31bff4d342
✓  Registered as : ohs_risk_score_model (Version 1)
   → Go to Machine Learning → Models to see it


## Cell 8 — Validate the Registered Model

Loads the model back and runs 2 contrasting test cases:
- **Case 1:** Critical severity + Stop Work Order + Not Complied With + Construction → expect very high score
- **Case 2:** Low severity + Forthwith Order + Complied With + Restaurant → expect low score

In [0]:
# Load the model back from the registry — same code path the serving endpoint uses
loaded_risk = mlflow.pyfunc.load_model("models:/workspace.default.ohs_risk_score_model/1")

# Two contrasting test cases to verify the model is working correctly
test_cases = pd.DataFrame([
    {
        # HIGH RISK CASE — Fatal investigation, Stop Work Order, not complied, Construction
        "FIELD_VISIT_DATE":  "2024-03-15",
        "FIELD_VISIT_TYPE":  "Field Visit",
        "CASE_TYPE":         "Investigation",     # reactive — incident already happened
        "CASE_STATUS":       "Open",
        "PRIMARY_NAICS":    "236110",             # Construction
        "CONTRAVENER_ROLE":  "Constructor",
        "ORDER_TYPE":        "Stop Work Order",   # most severe order type
        "ORDER_STATUS":      "Not Complied With", # employer has NOT fixed the issue
        "CASE_ACT":          "Occupational Health and Safety Act",
        "ACT_REG_ID":        "REG_213",
        "SEC":               "25",
        "SUBSEC":            "(1)",
        "CLAUSE":            "(a)",
        "SEVERITY_LEVEL":    "Critical",          # from Model 1 output
    },
    {
        # LOW RISK CASE — Routine inspection, forthwith order, complied, Restaurant
        "FIELD_VISIT_DATE":  "2024-06-01",
        "FIELD_VISIT_TYPE":  "Field Visit",
        "CASE_TYPE":         "Inspection",        # proactive routine visit
        "CASE_STATUS":       "Closed",            # case already resolved
        "PRIMARY_NAICS":    "722511",             # Restaurant
        "CONTRAVENER_ROLE":  "Employer",
        "ORDER_TYPE":        "Forthwith Order",   # least severe order type
        "ORDER_STATUS":      "Complied With",     # employer has fixed the issue
        "CASE_ACT":          "Occupational Health and Safety Act",
        "ACT_REG_ID":        "REG_851",
        "SEC":               "25",
        "SUBSEC":            "",
        "CLAUSE":            "",
        "SEVERITY_LEVEL":    "Low",               # from Model 1 output
    },
])

results = loaded_risk.predict(test_cases)

print("=== VALIDATION PREDICTIONS ===\n")
labels = ["HIGH RISK case (Construction, Stop Work Order, Not Complied)",
          "LOW RISK case  (Restaurant, Forthwith Order, Complied With)"]

for i, (r, label) in enumerate(zip(results, labels)):
    icon = {4: "🔴", 3: "🟠", 2: "🟡", 1: "🟢"}.get(r["risk_level_int"], "⚪")
    print(f"{icon}  {label}")
    print(f"    Risk Category : {r['risk_category']}")
    print(f"    Risk Score    : {r['risk_score']} / 100")
    print()

print("✓  Model validation complete — ready to create the serving endpoint")

=== VALIDATION PREDICTIONS ===

🔴  HIGH RISK case (Construction, Stop Work Order, Not Complied)
    Risk Category : Critical Risk
    Risk Score    : 98.0 / 100

🟢  LOW RISK case  (Restaurant, Forthwith Order, Complied With)
    Risk Category : Low Risk
    Risk Score    : 11.3 / 100

✓  Model validation complete — ready to create the serving endpoint
